In [23]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.express as px 


from sklearn.preprocessing import  StandardScaler , MinMaxScaler
from sklearn.metrics import  mean_squared_error , mean_absolute_error
from sklearn.linear_model import  LinearRegression , Ridge
from sklearn.model_selection import  GridSearchCV , cross_val_score , RandomizedSearchCV


from catboost import  CatBoostRegressor
from xgboost import  XGBRegressor

from sklearn.metrics import make_scorer
from sklearn.pipeline import  Pipeline

In [24]:
train = pd.read_csv('/home/nex/Downloads/calorie_expenditure/train.csv')
test = pd.read_csv('/home/nex/Downloads/calorie_expenditure/test.csv')

In [25]:
train.head()

,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,0,male,36,189.0,82.0,26.0,101.0,41.0,150.0
1,1,female,64,163.0,60.0,8.0,85.0,39.7,34.0
2,2,female,51,161.0,64.0,7.0,84.0,39.8,29.0
3,3,male,20,192.0,90.0,25.0,105.0,40.7,140.0
4,4,female,38,166.0,61.0,25.0,102.0,40.6,146.0


In [26]:
train = train.rename(columns={
    "id":"id",
    "Sex": "sex",
    "Age": "age",
    "Height": "height",
    "Weight": "weight",
    "Duration": "duration",
    "Heart_Rate": "heart_rate",
    "Body_Temp": "body_temp",
    "Calories": "calories"
})


In [27]:
train.columns

Index(['id', 'sex', 'age', 'height', 'weight', 'duration', 'heart_rate',
       'body_temp', 'calories'],
      dtype='object')

In [28]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   id          750000 non-null  int64  
 1   sex         750000 non-null  object 
 2   age         750000 non-null  int64  
 3   height      750000 non-null  float64
 4   weight      750000 non-null  float64
 5   duration    750000 non-null  float64
 6   heart_rate  750000 non-null  float64
 7   body_temp   750000 non-null  float64
 8   calories    750000 non-null  float64
dtypes: float64(6), int64(2), object(1)
memory usage: 51.5+ MB


In [29]:
train.isnull().sum()

id            0
sex           0
age           0
height        0
weight        0
duration      0
heart_rate    0
body_temp     0
calories      0
dtype: int64

In [30]:
train.duplicated().sum()

np.int64(0)

In [31]:
train.describe()

,id,age,height,weight,duration,heart_rate,body_temp,calories
count,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000,750000.000000
mean,374999.500000,41.420404,174.697685,75.145668,15.421015,95.483995,40.036253,88.282781
std,216506.495284,15.175049,12.824496,13.982704,8.354095,9.449845,0.779875,62.395349
min,0.000000,20.000000,126.000000,36.000000,1.000000,67.000000,37.100000,1.000000
25%,187499.750000,28.000000,164.000000,63.000000,8.000000,88.000000,39.600000,34.000000
50%,374999.500000,40.000000,174.000000,74.000000,15.000000,95.000000,40.300000,77.000000
75%,562499.250000,52.000000,185.000000,87.000000,23.000000,103.000000,40.700000,136.000000
max,749999.000000,79.000000,222.000000,132.000000,30.000000,128.000000,41.500000,314.000000


In [32]:
train["bmi"] = train["weight"]/(train["height"]/(100)**2)
train['sex'] = train['sex'].map({'male': 1, 'female': 0})

In [33]:
X_train = train.drop(columns = ["calories","id"],axis=1)
y_train = train["calories"]

In [34]:
xgb_pipe = Pipeline([
    ("scaler1",StandardScaler()),
    ("model1",XGBRegressor())
])


cat_pipe = Pipeline([
    ("scaler2",StandardScaler()),
    ("model2",CatBoostRegressor())
])



In [35]:
def rmsle(y_true, y_pred):
    y_true = np.maximum(0, y_true)
    y_pred = np.maximum(0, y_pred)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true))**2))

rmsle_scorer = make_scorer(rmsle, greater_is_better=False)


In [36]:
cat_params = {
    'model2__iterations': [100, 200],
    'model2__depth': [4, 6, 8],
    'model2__learning_rate': [0.01, 0.1, 0.2],
    'model2__l2_leaf_reg': [1, 3, 5]
}

xgb_params = {
    'model1__n_estimators': [100, 200],
    'model1__max_depth': [3, 6, 8],
    'model1__learning_rate': [0.01, 0.1, 0.2],
    'model1__reg_lambda': [0.5, 1, 2]
}


cat_search = RandomizedSearchCV(
    cat_pipe,
    cat_params,
    n_iter=10, 
    scoring=rmsle_scorer, 
    cv=5,
    verbose=1, 
    n_jobs=-1,
    random_state=42
)


xgb_search = RandomizedSearchCV(
    xgb_pipe,
    xgb_params,
    n_iter=10,
    scoring = rmsle_scorer,
    cv=5,
    verbose=1, 
    n_jobs=-1,
    random_state=42
)




In [37]:
xgb_search.fit(X_train,y_train)
cat_search.fit(X_train,y_train)



cat_rmsle = -cat_search.best_score_
xgb_rmsle = -xgb_search.best_score_

print("CatBoost RMSLE:", cat_rmsle)
print("XGBoost RMSLE:", xgb_rmsle)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Fitting 5 folds for each of 10 candidates, totalling 50 fits
0:	learn: 56.6043463	total: 991ms	remaining: 1m 38s
0:	learn: 56.5629235	total: 1.01s	remaining: 1m 39s
0:	learn: 56.5687116	total: 775ms	remaining: 1m 16s
0:	learn: 56.4855722	total: 1.49s	remaining: 4m 55s
1:	learn: 51.3705875	total: 1.54s	remaining: 1m 15s
1:	learn: 51.3748792	total: 1.03s	remaining: 50.5s
1:	learn: 51.4134619	total: 1.8s	remaining: 1m 28s
0:	learn: 56.6055169	total: 1.27s	remaining: 2m 6s
2:	learn: 46.6343476	total: 1.84s	remaining: 59.5s
1:	learn: 51.1594844	total: 1.9s	remaining: 3m 8s
2:	learn: 46.6376396	total: 1.37s	remaining: 44.4s
0:	learn: 56.5912294	total: 852ms	remaining: 1m 24s
2:	learn: 46.6753010	total: 2.22s	remaining: 1m 11s
0:	learn: 56.5204946	total: 1.07s	remaining: 3m 32s
3:	learn: 42.3924188	total: 2.19s	remaining: 52.6s
0:	learn: 56.4845593	total: 1.22s	remaining: 4m 2s
1:	learn: 51.4126960	total: 1.66s	remaining: 1m 21s
2:	

Exception ignored in: <function ResourceTracker.__del__ at 0x7f5ea958ea20>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


198:	learn: 3.4755967	total: 1m 29s	remaining: 449ms
98:	learn: 24.8928047	total: 48.7s	remaining: 49.7s
102:	learn: 24.0241718	total: 51.1s	remaining: 48.1s
121:	learn: 3.5823425	total: 51.7s	remaining: 33.1s
96:	learn: 25.3694111	total: 48.4s	remaining: 51.4s
122:	learn: 3.5805034	total: 52.1s	remaining: 32.6s
115:	learn: 3.5863251	total: 52.3s	remaining: 37.8s
103:	learn: 23.8106879	total: 51.4s	remaining: 47.5s
199:	learn: 3.4749021	total: 1m 29s	remaining: 0us
99:	learn: 24.6698200	total: 49.3s	remaining: 49.3s
104:	learn: 23.5993926	total: 51.7s	remaining: 46.8s
97:	learn: 25.1431742	total: 49s	remaining: 51s
116:	learn: 3.5848924	total: 52.7s	remaining: 37.4s
123:	learn: 3.5786444	total: 52.5s	remaining: 32.2s


/usr/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


100:	learn: 24.4509420	total: 49.7s	remaining: 48.7s
105:	learn: 23.3900557	total: 52s	remaining: 46.1s


...

178:	learn: 12.6360377	total: 1m 28s	remaining: 10.4s
187:	learn: 11.7749255	total: 1m 31s	remaining: 5.81s
80:	learn: 29.3117177	total: 39.3s	remaining: 57.8s
105:	learn: 25.5477513	total: 33.1s	remaining: 29.3s
193:	learn: 11.2345992	total: 1m 29s	remaining: 2.75s
0:	learn: 61.8084153	total: 229ms	remaining: 45.6s
73:	learn: 31.2380079	total: 34.8s	remaining: 59.2s
106:	learn: 25.3509757	total: 33.4s	remaining: 29s
81:	learn: 29.0457149	total: 39.7s	remaining: 57.1s
179:	learn: 12.5369907	total: 1m 28s	remaining: 9.87s
188:	learn: 11.6843161	total: 1m 31s	remaining: 5.33s
107:	learn: 25.1557006	total: 33.7s	remaining: 28.7s
1:	learn: 61.2500939	total: 644ms	remaining: 1m 3s
194:	learn: 11.1499000	total: 1m 29s	remaining: 2.29s
82:	learn: 28.7812485	total: 40.1s	remaining: 56.6s
74:	learn: 30.9516891	total: 35.4s	remaining: 59s
108:	learn: 24.9628708	total: 34s	remaining: 28.4s
189:	learn: 11.5945429	total: 1m 32s	remaining: 4.85s
2:	learn: 60.6970406	total: 1.04s	remaining: 1m 8s
180

Exception ignored in: <function ResourceTracker.__del__ at 0x7f791228ea20>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


13:	learn: 54.9758982	total: 4.17s	remaining: 55.5s
81:	learn: 29.0363527	total: 38.8s	remaining: 55.9s
197:	learn: 10.9142246	total: 1m 35s	remaining: 964ms
2:	learn: 60.7432630	total: 1.11s	remaining: 1m 12s
118:	learn: 23.1446301	total: 37.5s	remaining: 25.5s
14:	learn: 54.4882999	total: 4.51s	remaining: 55.6s
89:	learn: 27.0138254	total: 43.9s	remaining: 53.7s
188:	learn: 11.6902455	total: 1m 33s	remaining: 5.41s
198:	learn: 10.8333250	total: 1m 35s	remaining: 482ms
3:	learn: 60.1943977	total: 1.45s	remaining: 1m 10s
15:	learn: 54.0062806	total: 4.79s	remaining: 55.1s
82:	learn: 28.7722849	total: 39.4s	remaining: 55.5s
119:	learn: 22.9728505	total: 37.9s	remaining: 25.2s
16:	learn: 53.5290395	total: 4.99s	remaining: 53.7s
4:	learn: 59.6509663	total: 1.7s	remaining: 1m 6s
90:	learn: 26.7730510	total: 44.4s	remaining: 53.2s
83:	learn: 28.5144767	total: 39.7s	remaining: 54.9s
5:	learn: 59.1121030	total: 1.83s	remaining: 59.3s
199:	learn: 10.7533291	total: 1m 36s	remaining: 0us
189:	le

Exception ignored in: <function ResourceTracker.__del__ at 0x7f5ced296a20>
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/usr/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


8:	learn: 57.5346622	total: 2.94s	remaining: 1m 2s
20:	learn: 51.6659891	total: 6.29s	remaining: 53.6s
124:	learn: 22.1381505	total: 39.5s	remaining: 23.7s
9:	learn: 57.0217043	total: 3.18s	remaining: 1m
86:	learn: 27.7460848	total: 41.1s	remaining: 53.4s
93:	learn: 26.0585379	total: 45.9s	remaining: 51.7s
192:	learn: 11.3366919	total: 1m 34s	remaining: 3.44s
21:	learn: 51.2113392	total: 6.64s	remaining: 53.8s
10:	learn: 56.5127997	total: 3.39s	remaining: 58.2s
125:	learn: 21.9759089	total: 39.8s	remaining: 23.4s
94:	learn: 25.8224358	total: 46.2s	remaining: 51.1s
22:	learn: 50.7634800	total: 6.96s	remaining: 53.5s
11:	learn: 56.0069396	total: 3.64s	remaining: 57.1s
193:	learn: 11.2502452	total: 1m 35s	remaining: 2.95s
126:	learn: 21.8142411	total: 40.2s	remaining: 23.1s
23:	learn: 50.3191578	total: 7.2s	remaining: 52.8s
87:	learn: 27.4961996	total: 41.8s	remaining: 53.2s
12:	learn: 55.5106110	total: 3.9s	remaining: 56.2s
95:	learn: 25.5937173	total: 46.6s	remaining: 50.5s
127:	learn: 

In [38]:
if cat_rmsle < xgb_rmsle:
    best_model = cat_search.best_estimator_
    best_name = "CatBoost"
else:
    best_model = xgb_search.best_estimator_
    best_name = "XGBoost"

print("Best Model Selected:", best_name)

Best Model Selected: XGBoost


In [39]:
test.head()


,id,Sex,Age,Height,Weight,Duration,Heart_Rate,Body_Temp
0,750000,male,45,177.0,81.0,7.0,87.0,39.8
1,750001,male,26,200.0,97.0,20.0,101.0,40.5
2,750002,female,29,188.0,85.0,16.0,102.0,40.4
3,750003,female,39,172.0,73.0,20.0,107.0,40.6
4,750004,female,30,173.0,67.0,16.0,94.0,40.5


In [43]:
train.columns

Index(['id', 'sex', 'age', 'height', 'weight', 'duration', 'heart_rate',
       'body_temp', 'calories', 'bmi'],
      dtype='object')

In [46]:
test = test.rename(columns={
    "Sex": "sex",
    "Age": "age",
    "Height": "height",
    "Weight": "weight",
    "Duration": "duration",
    "Heart_Rate": "heart_rate",
    "Body_Temp": "body_temp",
}) 
test["bmi"] = test["weight"]/(test["height"]/(100)**2)
test["sex"] = test["sex"].map({'male': 1, 'female': 0})


X_test = test.drop(columns=["id"])

In [47]:
predictions = best_model.predict(X_test)

In [48]:
predictions

array([ 27.685099, 108.42111 ,  86.41041 , ...,  73.17721 , 168.84088 ,
        77.332245], shape=(250000,), dtype=float32)

In [49]:
# submission file 

submission = pd.DataFrame({
    "id": test["id"],
    "Calories": predictions
})


In [50]:
submission

,id,Calories
0,750000,27.685099
1,750001,108.421112
2,750002,86.410408
3,750003,126.440765
4,750004,76.623466
...,...,...
249995,999995,26.063120
249996,999996,9.209196
249997,999997,73.177208
249998,999998,168.840881


In [51]:
submission.to_csv('submission.csv', index=False)


In [52]:
print(xgb_rmsle)

0.06179780223199775
